In [1]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
import seaborn as sns
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

class HEMSEnvRealisticScheduling(gym.Env):
    """
    The definitive HEMS environment. It uses real, disaggregated load data to create a dynamic
    base load and defines schedulable tasks based on the remaining high-power appliances.
    """
    metadata = {"render.modes": ["human"]}

    def __init__(self, config):
        super().__init__()
        self.solar_df = pd.read_csv(config["solar_path"], parse_dates=["time"])
        self.load_df = pd.read_csv(config["load_path"], parse_dates=["Date_Time"])
        self.solar_df.columns = self.solar_df.columns.str.strip()
        self.load_df.columns = self.load_df.columns.str.strip()
        
        self.solar_panel_area = config["solar_panel_area"]
        self.max_battery = float(config["max_battery_kwh"])
        self.appliances = config["appliances"]
        
        self.battery_charge_eff = config.get("battery_charge_eff", 0.95)
        self.battery_discharge_eff = config.get("battery_discharge_eff", 0.95)
        self.battery_deg_cost_per_kwh = config.get("battery_deg_cost_per_kwh", 0.01)
        
        self.n_steps = min(len(self.load_df), len(self.solar_df))
        self.dt_hours = 1 / 60.0 # Data is per-minute
        self.steps_per_day = 24 * 60

        self.action_space = spaces.Discrete(1 + len(self.appliances))
        
        num_obs_features = 4 + len(self.appliances)
        obs_low = np.array([-1.0] * num_obs_features, dtype=np.float32)
        obs_high = np.array([1.0] * num_obs_features, dtype=np.float32)
        obs_high[0] = 1e6 # solar
        self.observation_space = spaces.Box(obs_low, obs_high, dtype=np.float32)
        
        self.time_step = 0
        self.battery_soc = 0.0
        self._reset_appliance_states()
        self.episode_info = []

    def _reset_appliance_states(self):
        self.appliance_states = []
        for app in self.appliances:
            self.appliance_states.append({
                "name": app["name"], "is_running": False,
                "steps_remaining": 0, "task_completed_today": False
            })

    def _compute_solar_output_kw(self, solar_row):
        ghi = pd.to_numeric(solar_row.get("ghi_pyr"), errors='coerce')
        humidity = pd.to_numeric(solar_row.get("relative_humidity"), errors='coerce')
        if pd.isna(ghi) or pd.isna(humidity): return 0.0
        panel_efficiency = 0.18
        weather_factor = max(0.0, 1.0 - (humidity / 100.0))
        pv_kw = (ghi * self.solar_panel_area * panel_efficiency * weather_factor) / 1000.0
        return max(pv_kw, 0.0)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.time_step = 0
        self.battery_soc = 0.5 * self.max_battery
        self._reset_appliance_states()
        self.episode_info = []
        obs, info = self._get_obs()
        return obs, info

    def _get_obs(self):
        idx = self.time_step % self.n_steps
        solar_row = self.solar_df.iloc[idx]
        
        solar_kw = self._compute_solar_output_kw(solar_row)
        soc_norm = self.battery_soc / self.max_battery if self.max_battery > 0 else 0.0
        
        hour = (self.time_step // 60) % 24
        hour_rad = (2.0 * math.pi * hour) / 24.0
        
        appliance_statuses = [1.0 if not s["task_completed_today"] else 0.0 for s in self.appliance_states]
        
        obs_list = [solar_kw, soc_norm, math.sin(hour_rad), math.cos(hour_rad)] + appliance_statuses
        return np.array(obs_list, dtype=np.float32), {}

    def _get_grid_price_for_hour(self, hour):
        return 46.85 if 19 <= hour <= 23 else 40.53

    def step(self, action):
        idx = self.time_step % self.n_steps
        current_load_row = self.load_df.iloc[idx]
        
        if self.time_step > 0 and self.time_step % self.steps_per_day == 0:
            for state in self.appliance_states:
                state["task_completed_today"] = False

        appliance_load_kw = 0
        for i, state in enumerate(self.appliance_states):
            if state["is_running"]:
                appliance_load_kw += self.appliances[i]["power_kw"]
                state["steps_remaining"] -= 1
                if state["steps_remaining"] == 0:
                    state["is_running"] = False
        
        if action > 0:
            app_idx = action - 1
            if not self.appliance_states[app_idx]["is_running"] and not self.appliance_states[app_idx]["task_completed_today"]:
                self.appliance_states[app_idx]["is_running"] = True
                self.appliance_states[app_idx]["steps_remaining"] = self.appliances[app_idx]["duration_steps"]
                self.appliance_states[app_idx]["task_completed_today"] = True
                appliance_load_kw += self.appliances[app_idx]["power_kw"]

        # *** CORRECTED BASE LOAD CALCULATION ***
        base_load_kw = current_load_row['Refrigerator_kW'] + current_load_row['WD_kW']
        total_load_kw = base_load_kw + appliance_load_kw
        total_load_kwh = total_load_kw * self.dt_hours
        
        solar_row = self.solar_df.iloc[idx]
        solar_kwh_available = self._compute_solar_output_kw(solar_row) * self.dt_hours
        
        grid_energy, solar_supplied, battery_supplied, battery_discharge = 0.0, 0.0, 0.0, 0.0
        remaining_load = total_load_kwh

        solar_supplied = min(solar_kwh_available, remaining_load)
        remaining_load -= solar_supplied
        if self.battery_soc > 0 and remaining_load > 0 and self.battery_discharge_eff > 0:
            discharge = min(self.battery_soc, remaining_load / self.battery_discharge_eff)
            battery_discharge, battery_supplied = discharge, discharge * self.battery_discharge_eff
            self.battery_soc -= discharge
            remaining_load -= battery_supplied
        if remaining_load > 0:
            grid_energy = remaining_load
        if solar_kwh_available > solar_supplied:
            charge = min((solar_kwh_available - solar_supplied) * self.battery_charge_eff, self.max_battery - self.battery_soc)
            self.battery_soc += charge

        hour = (self.time_step // 60) % 24
        grid_price = self._get_grid_price_for_hour(hour)
        grid_cost = grid_energy * grid_price
        battery_deg_cost = battery_discharge * self.battery_deg_cost_per_kwh
        
        reward = -(grid_cost + battery_deg_cost)
        
        terminated = False
        missed_task_today = False
        if (self.time_step + 1) % self.steps_per_day == 0:
            for state in self.appliance_states:
                if not state["task_completed_today"]:
                    terminated = True
                    missed_task_today = True
                    break
        
        if not terminated:
            terminated = self.time_step >= (self.n_steps - 1) 

        info = { "total_cost": grid_cost + battery_deg_cost, "task_missed": missed_task_today }
        self.episode_info.append(info)
        
        self.time_step += 1
        obs, _ = self._get_obs()
        return obs, reward, terminated, False, info

def train_scheduler(env_config, training_timesteps=50000, model_name="ppo_scheduler"):
    print(f"--- Training scheduler with config ---")
    def make_env():
        return HEMSEnvRealisticScheduling(env_config)

    vec_env = DummyVecEnv([make_env for _ in range(4)])
    vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=False, clip_obs=10.0)
    
    model = PPO("MlpPolicy", vec_env, policy_kwargs=dict(net_arch=dict(pi=[128, 128], vf=[128, 128])), verbose=0)
    model.learn(total_timesteps=training_timesteps, progress_bar=True)
    
    model.save(model_name)
    vec_env.save(f"{model_name}_vecnormalize.pkl")
    print("--- Training Complete ---")
    return model_name

def evaluate_scheduler(model_name, env_config):
    print(f"\n--- Evaluating scheduler ---")
    model = PPO.load(model_name)
    
    def make_env():
        return HEMSEnvRealisticScheduling(env_config)

    eval_vec_env = DummyVecEnv([make_env])
    eval_vec_env = VecNormalize.load(f"{model_name}_vecnormalize.pkl", eval_vec_env)
    eval_vec_env.training = False
    eval_vec_env.norm_reward = False
    
    all_infos = []
    obs = eval_vec_env.reset()
    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, _, terminated, info = eval_vec_env.step(action)
        all_infos.append(info[0])
        done = terminated[0]

    return pd.DataFrame(all_infos)

if __name__ == "__main__":
    
    load_df_analysis = pd.read_csv("House31.csv")
    load_df_analysis.columns = load_df_analysis.columns.str.strip()
    ac_power = load_df_analysis[['AC_LR_kW', 'AC_DR_kW', 'AC_BR_kW']].sum(axis=1)
    
    avg_ac_power_when_on = ac_power[ac_power > 0.1].mean()
    ac_duration_minutes = 120
    
    env_config = {
        "load_path": "House31.csv",
        "solar_path": "solar.csv", 
        "solar_panel_area": 27.8,
        "max_battery_kwh": 15.0, # Using 15kWh battery
        "appliances": [
            {
                "name": "AirConditioning", 
                "power_kw": avg_ac_power_when_on, 
                "duration_steps": int(ac_duration_minutes / 1) # 1-minute steps
            }
        ]
    }
    
    model_path = train_scheduler(env_config, training_timesteps=200000, model_name="final_scheduler")
    results = evaluate_scheduler(model_path, env_config)
    
    total_cost = results['total_cost'].sum()
    missed_tasks = results['task_missed'].sum()
    num_days = len(results) / (24 * 60)

    print("\n=== Definitive Appliance Scheduling Summary ===")
    print(f"Total Simulation Days: {num_days:.0f}")
    print(f"Total Annual Energy Cost: {total_cost:,.2f} PKR")
    print(f"Total Days with Missed Tasks: {missed_tasks:.0f}")

--- Training scheduler with config ---


--- Training Complete ---

--- Evaluating scheduler ---

=== Definitive Appliance Scheduling Summary ===
Total Simulation Days: 92
Total Annual Energy Cost: 1,530.66 PKR
Total Days with Missed Tasks: 0
